In [ ]:
"""
Logistics Database — Schema & Load SQL Generator
====================================================

Generates PostgreSQL CREATE TABLE + COPY statements directly from your
cleaned CSVs, so column order and types always match the file exactly.


Because the cleaning script already parsed date columns to real
datetime dtype, they'll write out of pandas as ISO-format strings
(YYYY-MM-DD or YYYY-MM-DD HH:MM:SS), which Postgres TIMESTAMP/DATE
columns parse natively via COPY — no post-load ::TIMESTAMP cast needed
this time, as long as the schema declares them as TIMESTAMP/DATE up front.

HOW TO USE
----------
1. Edit CLEAN_FOLDER and OUTPUT_FOLDER below.
2. Run: python generate_schema.py
3. It writes:
     sql/01_create_schema.sql
     sql/02_load_data.sql
     sql/03_validate_load.sql
4. Review 01_create_schema.sql before running it 
   VARCHAR lengths and NUMERIC precision look sane for your data.
5. Run the three .sql files 
"""

import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG — edit to match your paths
# ============================================================

CLEAN_FOLDER = Path(r"C:\Projects\3Year Truck Operations\data\clean")
OUTPUT_FOLDER = Path(r"C:\Projects\3Year Truck Operations\sql")

FILES = {
    "drivers": "drivers_clean.csv",
    "trucks": "trucks_clean.csv",
    "trailers": "trailers_clean.csv",
    "customers": "customers_clean.csv",
    "facilities": "facilities_clean.csv",
    "routes": "routes_clean.csv",
    "loads": "loads_clean.csv",
    "trips": "trips_clean.csv",
    "fuel_purchases": "fuel_purchases_clean.csv",
    "maintenance_records": "maintenance_records_clean.csv",
    "delivery_events": "delivery_events_clean.csv",
    "safety_incidents": "safety_incidents_clean.csv",
    "driver_monthly_metrics": "driver_monthly_metrics_clean.csv",
    "truck_utilization_metrics": "truck_utilization_metrics_clean.csv",
}

PRIMARY_KEYS = {
    "drivers": "driver_id",
    "trucks": "truck_id",
    "trailers": "trailer_id",
    "customers": "customer_id",
    "facilities": "facility_id",
    "routes": "route_id",
    "loads": "load_id",
    "trips": "trip_id",
    "fuel_purchases": "fuel_purchase_id",
    "maintenance_records": "maintenance_id",
    "delivery_events": "event_id",
    "safety_incidents": "incident_id",
    # composite-key tables handled separately below
}

COMPOSITE_KEYS = {
    "driver_monthly_metrics": ["driver_id", "month"],
    "truck_utilization_metrics": ["truck_id", "month"],
}

# (child_table, fk_column, parent_table, parent_pk)
FOREIGN_KEYS = [
    ("loads", "customer_id", "customers", "customer_id"),
    ("loads", "route_id", "routes", "route_id"),
    ("trips", "load_id", "loads", "load_id"),
    ("trips", "driver_id", "drivers", "driver_id"),
    ("trips", "truck_id", "trucks", "truck_id"),
    ("trips", "trailer_id", "trailers", "trailer_id"),
    ("fuel_purchases", "trip_id", "trips", "trip_id"),
    ("fuel_purchases", "truck_id", "trucks", "truck_id"),
    ("fuel_purchases", "driver_id", "drivers", "driver_id"),
    ("maintenance_records", "truck_id", "trucks", "truck_id"),
    ("delivery_events", "load_id", "loads", "load_id"),
    ("delivery_events", "trip_id", "trips", "trip_id"),
    ("delivery_events", "facility_id", "facilities", "facility_id"),
    ("safety_incidents", "trip_id", "trips", "trip_id"),
    ("safety_incidents", "truck_id", "trucks", "truck_id"),
    ("safety_incidents", "driver_id", "drivers", "driver_id"),
    ("driver_monthly_metrics", "driver_id", "drivers", "driver_id"),
    ("truck_utilization_metrics", "truck_id", "trucks", "truck_id"),
]

# Load order respects FK dependencies — parents before children
LOAD_ORDER = [
    "drivers", "trucks", "trailers", "customers", "facilities", "routes",
    "loads", "trips", "fuel_purchases", "maintenance_records",
    "delivery_events", "safety_incidents",
    "driver_monthly_metrics", "truck_utilization_metrics",
]

# Columns known to be dates/timestamps (after cleaning, these are real
# datetime dtype in the CSV, so declare them as TIMESTAMP/DATE directly)
DATE_COLUMNS = {
    "hire_date": "DATE", "termination_date": "DATE", "date_of_birth": "DATE",
    "acquisition_date": "DATE", "contract_start_date": "DATE",
    "load_date": "DATE", "dispatch_date": "DATE", "purchase_date": "DATE",
    "maintenance_date": "DATE", "incident_date": "DATE",
    "scheduled_datetime": "TIMESTAMP", "actual_datetime": "TIMESTAMP",
    "month": "DATE",
}


# ============================================================
# TYPE MAPPING
# ============================================================

def pg_type_for(col_name, series):
    if col_name in DATE_COLUMNS:
        return DATE_COLUMNS[col_name]

    dtype = series.dtype
    if pd.api.types.is_integer_dtype(dtype):
        return "INTEGER"
    if pd.api.types.is_float_dtype(dtype):
        return "NUMERIC(12,2)"
    if pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"

    # object/text — size VARCHAR to the observed max length with headroom,
    # fall back to TEXT for anything long (e.g. incident descriptions)
    max_len = series.dropna().astype(str).map(len).max() if series.notna().any() else 0
    if max_len == 0 or pd.isna(max_len):
        return "VARCHAR(255)"
    if max_len > 500:
        return "TEXT"
    return f"VARCHAR({int(max_len * 2) + 10})"


# ============================================================
# SCHEMA GENERATION
# ============================================================

def generate_schema(tables):
    lines = ["-- Auto-generated from cleaned CSVs. Review VARCHAR/NUMERIC sizing before running.\n"]

    for name in LOAD_ORDER:
        if name not in tables:
            continue
        df = tables[name]
        lines.append(f"DROP TABLE IF EXISTS {name} CASCADE;")
        lines.append(f"CREATE TABLE {name} (")

        col_defs = []
        for col in df.columns:
            pg_type = pg_type_for(col, df[col])
            col_defs.append(f"    {col} {pg_type}")

        pk = PRIMARY_KEYS.get(name)
        composite_pk = COMPOSITE_KEYS.get(name)
        if pk:
            col_defs.append(f"    PRIMARY KEY ({pk})")
        elif composite_pk:
            col_defs.append(f"    PRIMARY KEY ({', '.join(composite_pk)})")

        lines.append(",\n".join(col_defs))
        lines.append(");\n")

    # Foreign keys added after all tables exist
    lines.append("-- Foreign key constraints")
    for child, fk_col, parent, parent_pk in FOREIGN_KEYS:
        if child not in tables or parent not in tables:
            continue
        constraint_name = f"fk_{child}_{fk_col}"
        lines.append(
            f"ALTER TABLE {child} ADD CONSTRAINT {constraint_name} "
            f"FOREIGN KEY ({fk_col}) REFERENCES {parent}({parent_pk});"
        )

    lines.append("\n-- Indexes on FK columns for join performance")
    for child, fk_col, parent, parent_pk in FOREIGN_KEYS:
        if child not in tables:
            continue
        lines.append(f"CREATE INDEX idx_{child}_{fk_col} ON {child}({fk_col});")

    return "\n".join(lines)


# ============================================================
# LOAD SQL GENERATION
# ============================================================

def generate_load_sql(tables):
    lines = ["-- COPY commands. Column order matches the CSV header exactly.",
             "-- Update the file path below to your actual clean CSV location.\n"]
    for name in LOAD_ORDER:
        if name not in tables:
            continue
        df = tables[name]
        cols = ", ".join(df.columns)
        filename = FILES[name]
        path_str = str((CLEAN_FOLDER / filename).as_posix())
        lines.append(
            f"COPY {name} ({cols})\n"
            f"FROM '{path_str}'\n"
            f"DELIMITER ',' CSV HEADER;\n"
        )
    return "\n".join(lines)


# ============================================================
# VALIDATE SQL GENERATION
# ============================================================

def generate_validate_sql(tables):
    lines = ["-- Row count sanity check against source CSVs\n"]
    for name in LOAD_ORDER:
        if name not in tables:
            continue
        expected = len(tables[name])
        lines.append(f"SELECT '{name}' AS table_name, COUNT(*) AS loaded_rows, "
                     f"{expected} AS expected_rows FROM {name}")
        lines.append("UNION ALL")
    if lines[-1] == "UNION ALL":
        lines[-1] = ";"  # replace trailing UNION ALL with terminator
    lines.append("\n-- FK orphan check (should return 0 rows for each)")
    for child, fk_col, parent, parent_pk in FOREIGN_KEYS:
        if child not in tables or parent not in tables:
            continue
        lines.append(
            f"SELECT '{child}.{fk_col}' AS relationship, COUNT(*) AS orphans\n"
            f"FROM {child} c LEFT JOIN {parent} p ON c.{fk_col} = p.{parent_pk}\n"
            f"WHERE c.{fk_col} IS NOT NULL AND p.{parent_pk} IS NULL;\n"
        )
    return "\n".join(lines)


# ============================================================
# MAIN
# ============================================================

def main():
    print("Reading cleaned CSVs from:", CLEAN_FOLDER.resolve())
    tables = {}
    for name, filename in FILES.items():
        path = CLEAN_FOLDER / filename
        if not path.exists():
            print(f"  [SKIP] {name}: not found at {path}")
            continue
        df = pd.read_csv(path, low_memory=False)
        # re-parse date columns so dtype-based typing works correctly
        for col in df.columns:
            if col in DATE_COLUMNS:
                df[col] = pd.to_datetime(df[col], errors="coerce")
        tables[name] = df

    print(f"Loaded {len(tables)}/{len(FILES)} tables.\n")

    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

    schema_sql = generate_schema(tables)
    load_sql = generate_load_sql(tables)
    validate_sql = generate_validate_sql(tables)

    (OUTPUT_FOLDER / "01_create_schema.sql").write_text(schema_sql, encoding="utf-8")
    (OUTPUT_FOLDER / "02_load_data.sql").write_text(load_sql, encoding="utf-8")
    (OUTPUT_FOLDER / "03_validate_load.sql").write_text(validate_sql, encoding="utf-8")

    print("Written:")
    print(f"  {OUTPUT_FOLDER / '01_create_schema.sql'}")
    print(f"  {OUTPUT_FOLDER / '02_load_data.sql'}")
    print(f"  {OUTPUT_FOLDER / '03_validate_load.sql'}")
    print("\nReview 01_create_schema.sql before running — check VARCHAR/NUMERIC sizing.")
    print("Run order: 01 (schema) -> 02 (load) -> 03 (validate).")


if __name__ == "__main__":
    main()

Reading cleaned CSVs from: C:\Projects\3Year Truck Operations\data\clean
Loaded 14/14 tables.

Written:
  C:\Projects\3Year Truck Operations\sql\01_create_schema.sql
  C:\Projects\3Year Truck Operations\sql\02_load_data.sql
  C:\Projects\3Year Truck Operations\sql\03_validate_load.sql

Review 01_create_schema.sql before running — check VARCHAR/NUMERIC sizing.
Run order: 01 (schema) -> 02 (load) -> 03 (validate).
